### Import Dependencies

In [2]:
import openai
import instructor

from pydantic import BaseModel, Field
from qdrant_client import QdrantClient

### Mock Example

In [3]:
prompt = """
You are a helpful assistant.
Return an answer to the question.
Question: what is your name?"""

In [4]:
response = openai.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

I’m ChatGPT.


### Add Instructor (Structured Outputs)

In [6]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [7]:
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [8]:
response = client.create(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [9]:
response

Answer(answer='I’m ChatGPT, an AI assistant.')

In [10]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [11]:
response

Answer(answer='I’m ChatGPT, an AI assistant.')

In [12]:
raw_response

Response(id='resp_07c1dd63f7fe014d0069f4e9816c808195a0a5c34d6d806427', created_at=1777658241.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"I’m ChatGPT, an AI assistant."}', call_id='call_NTDo9ecsT7TSVxJCfMkARulh', name='Answer', type='function_call', id='fc_07c1dd63f7fe014d0069f4e981b99c8195ae1411062b5b3cbc', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question.', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description=None)], top_p=0.98, background=False, completed_at=1777658241.0, conversation=None, max_output_tokens=

In [13]:
class AnswerWithReasoning(BaseModel):
    answer: str = Field(description="Answer to the question.")
    reasoning: str = Field(description="Reasoning for the answer.")

In [14]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=AnswerWithReasoning
)

In [15]:
response

AnswerWithReasoning(answer='I’m ChatGPT.', reasoning='The user asked for my name; as an AI assistant, my name is ChatGPT.')

### RAG Pipeline

In [19]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [20]:
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response

def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result

In [21]:
output = rag_pipeline("Can you suggest me some earphones?")

In [22]:
output

{'data_object': RAGGenerationResponse(answer='Sure—here are some earphones from the available products:\n\n- **B0CBMPG524 (Open-ear true wireless, Bluetooth 5.3)** — up to **60H** with charging case, **IPX7** waterproof, earhooks for workouts, and lets you hear your surroundings.\n- **B0C6K1GQCF (Wireless earbuds, Bluetooth 5.1)** — **30H** with case, **IPX7** waterproof, **2-mic** noise-canceling for clearer calls, touch controls.\n- **B0B9FTVL58 (Wireless earbuds, Bluetooth 5.3)** — **37H** playback, **IPX7** waterproof, deep bass, smart touch, with charging case.\n- **B0BNHVLF7G (Bone conduction open-ear, Bluetooth 5.3)** — sweat resistant (**IP56**), open-ear comfort, with mic for calls, **8H** battery + quick charge.\n- **B0C142QS8X (Kids wired over-ear, 3.5mm)** — volume limited to **94dB** for hearing protection, foldable, padded, and works with most 3.5mm devices.\n\nTell me what you need (in-ear vs open-ear, waterproof level, budget, and whether it’s for kids/phone calls), and

In [23]:
print(output["answer"])

Sure—here are some earphones from the available products:

- **B0CBMPG524 (Open-ear true wireless, Bluetooth 5.3)** — up to **60H** with charging case, **IPX7** waterproof, earhooks for workouts, and lets you hear your surroundings.
- **B0C6K1GQCF (Wireless earbuds, Bluetooth 5.1)** — **30H** with case, **IPX7** waterproof, **2-mic** noise-canceling for clearer calls, touch controls.
- **B0B9FTVL58 (Wireless earbuds, Bluetooth 5.3)** — **37H** playback, **IPX7** waterproof, deep bass, smart touch, with charging case.
- **B0BNHVLF7G (Bone conduction open-ear, Bluetooth 5.3)** — sweat resistant (**IP56**), open-ear comfort, with mic for calls, **8H** battery + quick charge.
- **B0C142QS8X (Kids wired over-ear, 3.5mm)** — volume limited to **94dB** for hearing protection, foldable, padded, and works with most 3.5mm devices.

Tell me what you need (in-ear vs open-ear, waterproof level, budget, and whether it’s for kids/phone calls), and I’ll recommend the best match.
